In [ ]:
# check available CUDA devices
import torch
devices = []
if torch.cuda.is_available():
        for i in range(torch.cuda.device_count()):
            device_name = torch.cuda.get_device_name(i)
            devices.append({
                "type": "CUDA",
                "available": True,
                "name": device_name,
                "index": i
            })
else:
    devices.append({"type": "CUDA", "available": False, "name": "N/A"})
devices

In [ ]:
# change folder into the root of the ASR project
import os

if not os.path.isdir("Configs"):
    %cd ../

!pwd

In [ ]:

# import packages, define helper utilities
import os
import yaml
import torch
import torch.nn as nn
import pandas as pd

from models import ASRCNN
from meldataset import build_dataloader
from token_map import build_token_map_from_data


def cfg_get_nested(cfg: dict, path, default=None, sep="."):
    """Get a nested value from a dict using a list of keys or a dot-separated string."""
    if isinstance(path, str):
        keys = path.split(sep) if path else []
    else:
        keys = path

    cur = cfg
    for k in keys:
        if isinstance(cur, dict) and k in cur:
            cur = cur[k]
        else:
            return default
    return cur


def prepare_model_params(config: dict, vocab_size: int) -> dict:
    defaults = {
        "input_dim": 80,
        "hidden_dim": 256,
        "n_token": vocab_size,
        "token_embedding_dim": 512,
        "n_layers": 5,
        "decoder_type": "conformer",
        "decoder_config": {},
    }

    raw_params = cfg_get_nested(config, "model_params", {}) or {}
    if not isinstance(raw_params, dict):
        raw_params = {}

    params = {
        **defaults,
        **{k: v for k, v in raw_params.items()
           if k not in {"decoder", "decoder_type", "decoder_config", "location_kernel_size"}},
    }

    params["n_token"] = int(raw_params.get("n_token", params.get("n_token", vocab_size)))
    if params["n_token"] <= 0:
        params["n_token"] = vocab_size

    decoder_type = raw_params.get("decoder_type")
    decoder_config = raw_params.get("decoder_config") or {}
    decoder_entry = raw_params.get("decoder")

    if isinstance(decoder_entry, dict):
        decoder_config = decoder_entry
        decoder_type = decoder_entry.get("type", decoder_type)

    if decoder_type is None and "location_kernel_size" in raw_params:
        decoder_type = "lstm"
    if decoder_type is None:
        decoder_type = defaults.get("decoder_type", "conformer")

    if decoder_type == "lstm":
        lstm_cfg = decoder_config.get("lstm", {})
        kernel = raw_params.get("location_kernel_size", lstm_cfg.get("location_kernel_size"))
        if kernel is not None:
            lstm_cfg = {**lstm_cfg, "location_kernel_size": kernel}
        if "random_mask_prob" in raw_params:
            lstm_cfg = {**lstm_cfg, "random_mask_prob": raw_params["random_mask_prob"]}
        decoder_config = {**decoder_config, "lstm": lstm_cfg}

    params["decoder_type"] = decoder_type if decoder_type in {"lstm", "conformer"} else "conformer"
    params["decoder_config"] = decoder_config

    return params


def load_token_map_from_config(config):
    token_src = config.get("phoneme_maps_path")
    if not token_src:
        return build_token_map_from_data(
            config.get("train_data"),
            config.get("val_data"),
            config.get("ood_data"),
            apply_asr_tokenizer=True,
        )
    if isinstance(token_src, dict):
        return token_src
    csv = pd.read_csv(token_src, header=None).values
    return {word: index for word, index in csv}


def load_asr_model(model_path, config_path, device):
    with open(config_path) as f:
        config = yaml.safe_load(f)

    token_map = load_token_map_from_config(config)

    model_params = prepare_model_params(config, len(token_map))

    model = ASRCNN(**model_params)
    checkpoint = torch.load(model_path, map_location="cpu", weights_only=False)
    state_dict = checkpoint.get("model", checkpoint)
    try:
        model.load_state_dict(state_dict)
    except RuntimeError:
        sanitized_state = {k.replace("module.", ""): v for k, v in state_dict.items()}
        model.load_state_dict(sanitized_state)

    model.to(device)
    model.eval()
    return model, config, token_map


def build_dev_dataloader(config, device, batch_size=None, num_workers=None):
    val_csv_path = config.get("val_data")
    if val_csv_path is None:
        raise ValueError("Validation CSV path ('val_data') not found in the config.")

    with open(val_csv_path, "r", encoding="utf-8") as f:
        raw_lines = [line.rstrip("
") for line in f]

    path_list = []
    for raw in raw_lines:
        if not raw.strip():
            continue
        parts = raw.split("|")
        if len(parts) == 1:
            continue
        path = parts[0]
        if len(parts) == 2:
            text = parts[1]
            speaker = ""
        else:
            text = "|".join(parts[1:-1])
            speaker = parts[-1]
        path_list.append([path, text, speaker])

    if batch_size is None:
        batch_size = int(cfg_get_nested(config, "eval_params.batch_size", cfg_get_nested(config, "batch_size", 4)))
    if num_workers is None:
        num_workers = int(cfg_get_nested(config, "dataloader_params.val_num_workers", 2))

    dataset_params = {
        "dict_path": cfg_get_nested(config, "phoneme_maps_path", "Data/word_index_dict.txt"),
        "sr": cfg_get_nested(config, "preprocess_params.sr", 24000),
        "spect_params": cfg_get_nested(
            config,
            "preprocess_params.spect_params",
            {"n_fft": 1024, "win_length": 1024, "hop_length": 300},
        ),
        "mel_params": cfg_get_nested(config, "preprocess_params.mel_params", {"n_mels": 80}),
    }

    collate_config = {"return_wave": False}
    device_flag = device.type if isinstance(device, torch.device) else str(device)
    loader = build_dataloader(
        path_list=path_list,
        validation=True,
        batch_size=batch_size,
        num_workers=num_workers,
        device=device_flag,
        collate_config=collate_config,
        dataset_config=dataset_params,
    )
    return loader



In [ ]:
checkpoint_dir = "Checkpoint"

files = [f for f in os.listdir( checkpoint_dir + "/") if f.startswith('epoch_') and f.endswith('.pth')]
sorted_files = sorted(files, key=lambda x: int(x.split('_')[-1].split('.')[0]))

model_path = checkpoint_dir + "/" + sorted_files[-1]
#model_path = "Checkpoint/epoch_00080.pth"

config_path = 'Checkpoint/config.yml'

config = yaml.safe_load(open(config_path))
phoneme_map = config.get('phoneme_maps_path')
if not phoneme_map:
    phoneme_map = build_token_map_from_data(config.get('train_data'), config.get('val_data'), config.get('ood_data'), apply_asr_tokenizer=True)

test_csv_path = config['val_data']

def _dict_desc(obj):
    return obj if isinstance(obj, str) else 'built from dataset'

print( "model --> " + model_path )
print( "config --> " + config_path)
print( "dictionary --> " + _dict_desc(phoneme_map))
print( "test: --> " + test_csv_path)

model = load_asr_model(model_path, config_path)
model.eval()

print( "All OK ✓")


In [ ]:
text_cleaner = TextCleaner(phoneme_map)
device = "cpu"

with open(test_csv_path, "r", encoding="utf-8") as f:
    test_lines = f.readlines()

dataset_params = {
        'dict_path': cfg_get_nested( config, 'phoneme_maps_path', 'Data/word_index_dict.txt'),
        'sr': cfg_get_nested( config, 'preprocess_params.sr', 24000),
        'spect_params': cfg_get_nested( config, 'preprocess_params.spect_params', {
            'n_fft': 1024,
            'win_length': 1024,
            'hop_length': 300
        }),
        'mel_params': cfg_get_nested( config, 'preprocess_params.mel_params', { 'n_mels': 80 })
    }

test_loader = build_dataloader(
    path_list=[l[:-1].split('|') for l in test_lines],
    validation=True,
    batch_size=1,
    num_workers=2,
    device=device,
    collate_config={"return_wave": False},
    dataset_config=dataset_params
)

if isinstance(phoneme_map, str):
    csv = pd.read_csv(phoneme_map, header=None).values
    wlist = {word: index for word, index in csv}
else:
    wlist = phoneme_map
index2phoneme = {v: k for k, v in wlist.items()}

predictions = []
references = []
cleartexts = []

model.eval()
log_interval = 10
total = len(test_lines)
cntr = 0
maxtestsize = 1
#maxtestsize = 0

with torch.no_grad():
    for batch in test_loader:
        cleartexts.append(test_lines[cntr])

        texts, input_lengths, mels, output_lengths = batch  # from Collater

        mels = mels.to(device)
        output = model(mels)
        predicted_ids = torch.argmax(output, dim=-1)

        print(f"Batch {cntr} - Expected text: {test_lines[cntr].strip().split('|')[1]}")
        predicted_text = ''.join([text_cleaner.inverse_mapping.get(phoneme.item(), '<unk>') for phoneme in predicted_ids[0]])
        print(f"Batch {cntr} - Predicted text: {predicted_text}")

        for i in range(predicted_ids.size(0)):
            pred_seq = predicted_ids[i][:output_lengths[i]//3]
            ref_seq = texts[i][:input_lengths[i]]

            pred_phonemes = [index2phoneme.get(p.item(), '') for p in pred_seq]
            ref_phonemes = [index2phoneme.get(r.item(), '') for r in ref_seq]

            predictions.append(pred_phonemes)
            references.append(ref_phonemes)

        cntr += 1
        if (cntr)%log_interval == 0:
            print(f"{cntr} of {total} sentences tested")

        if maxtestsize > 0 and cntr >= maxtestsize:
            print(f"early stop reached at {maxtestsize} sentences")
            break

print("Done - all sententes tested.")


In [ ]:
# Clean extra quotes and join tokens into space-separated strings
references_cleaned = [' '.join(token.strip('"') for token in seq) for seq in references]
predictions_cleaned = [' '.join(token.strip('"') for token in seq) for seq in predictions]

# Now use jiwer
per = wer(references_cleaned, predictions_cleaned)
print(f'Phoneme Error Rate: {per:.4f}')

In [ ]:
###########################################
# Find the best AUX model (with best PER) #
###########################################

#checkpoint_dir = "Checkpoint-en-test"

files = [f for f in os.listdir( checkpoint_dir + "/") if f.startswith('epoch_') and f.endswith('.pth')]
sorted_files = sorted(files, key=lambda x: int(x.split('_')[-1].split('.')[0]))

model_path = checkpoint_dir + "/" + sorted_files[-1]
#model_path = "Checkpoint/epoch_00040.pth"
#config_path = "Configs/config.yml"

config = yaml.safe_load(open(config_path))
phoneme_map = config.get('phoneme_maps_path')
if not phoneme_map:
    phoneme_map = build_token_map_from_data(config.get('train_data'), config.get('val_data'), config.get('ood_data'), apply_asr_tokenizer=True)

test_csv_path = config['val_data']

def _dict_desc(obj):
    return obj if isinstance(obj, str) else 'built from dataset'

print( "config --> " + config_path )
print( "dictionary --> " + _dict_desc(phoneme_map) )
print( "test --> " + test_csv_path )

text_cleaner = TextCleaner(phoneme_map)
#text_cleaner = TextCleaner()
device = "cpu"

with open(test_csv_path, "r", encoding="utf-8") as f:
    test_lines = f.readlines()

dataset_params = {
        'dict_path': cfg_get_nested( config, 'phoneme_maps_path', 'Data/word_index_dict.txt'),
        'sr': cfg_get_nested( config, 'preprocess_params.sr', 24000),
        'spect_params': cfg_get_nested( config, 'preprocess_params.spect_params', {
            'n_fft': 1024,
            'win_length': 1024,
            'hop_length': 300
        }),
        'mel_params': cfg_get_nested( config, 'preprocess_params.mel_params', { 'n_mels': 80 })
    }

test_loader = build_dataloader(
    #path_list=test_lines,
    path_list=[l[:-1].split('|') for l in test_lines],
    validation=True,
    batch_size=1,
    num_workers=2,
    device=device,
    collate_config={"return_wave": False},
    dataset_config=dataset_params
)

if isinstance(phoneme_map, str):
    csv = pd.read_csv(phoneme_map, header=None).values
    wlist = {word: index for word, index in csv}
else:
    wlist = phoneme_map
index2phoneme = {v: k for k, v in wlist.items()}

best_model = ""
best_model_per = 100.0

model_cntr = 1
total_files = len(sorted_files)
results = []
#maxtestsize = 0 # will test the full validation set - may take a while and is usually not necessary
maxtestsize = 25

for aux_model_file in sorted_files:
    model_path = checkpoint_dir + "/" + aux_model_file
    model = load_asr_model(model_path, config_path)
    model.eval()

    print(f"[{model_cntr}/{total_files}] Now evaluating AUX model: {aux_model_file}")
    model_cntr += 1

    predictions = []
    references = []
    first_ref = ""
    first_pred = ""

    test_iter = iter(test_loader)
    with torch.no_grad():
        for sample_idx in range(len(test_loader)):
            if maxtestsize > 0 and sample_idx >= maxtestsize:
                break

            batch = next(test_iter)
            texts, input_lengths, mels, output_lengths = batch
            mels = mels.to(device)
            output = model(mels)
            predicted_ids = torch.argmax(output, dim=-1)

            for i in range(predicted_ids.size(0)):
                pred_seq = predicted_ids[i][:output_lengths[i] // 3]
                ref_seq = texts[i][:input_lengths[i]]

                pred_phonemes = [index2phoneme.get(p.item(), '') for p in pred_seq]
                ref_phonemes = [index2phoneme.get(r.item(), '') for r in ref_seq]

                predictions.append(pred_phonemes)
                references.append(ref_phonemes)

                if first_ref == "":
                    first_ref = ' '.join(ref_phonemes)
                    first_pred = ' '.join(pred_phonemes)

    references_cleaned = [' '.join(seq) for seq in references]
    predictions_cleaned = [' '.join(seq) for seq in predictions]
    per = wer(references_cleaned, predictions_cleaned)

    print(f'Phoneme Error Rate: {per:.4f} {"✓" if per < best_model_per else "✗"}')
    if per < best_model_per:
        best_model_per = per
        best_model = aux_model_file

    results.append({
        'model': aux_model_file,
        'per': per,
        'first_ref': first_ref,
        'first_pred': first_pred
    })

results_sorted = sorted(results, key=lambda x: x['per'])

print("===================")
print("PERFORMANCE SUMMARY")
print("===================")
for res in results_sorted:
    print(f"Model: {res['model']}")
    print(f"PER: {res['per']:.4f}")
    print(f"Reference: {res['first_ref']}")
    print(f"Prediction: {res['first_pred']}")
    print("------------------------")

best = results_sorted[0]
print(f"✅ Best model: {best['model']} with PER = {best['per']:.4f}")
